In [37]:
from dataloader import *
from augmentations import *
from models import *
from training import *
import matplotlib.pyplot as plt
from itertools import product
from statistics import mean, stdev

## Experiment loop for testing 

In [29]:
params = {
     #training params
    "batch_size": [8, 32, 128],
    "learning_rate": [1e-3, 1e-4, 1e-5],
    "optimizer": ['Adam', 'SGD'],
    #regularization params
    "dropout": [0.2, 0.5],
    "weight_decay": [0, 1e-3, 0.5]
}

# Create experiment grid
experiment_grid = [dict(zip(params.keys(), v)) for v in product(*params.values())]

In [30]:
# Only for testing!!!!!!
from torch.utils.data import Subset
import numpy as np

def get_subset(dataset, fraction=0.1):
    size = int(len(dataset) * fraction)
    indices = np.random.choice(len(dataset), size, replace=False)
    return Subset(dataset, indices)

In [31]:
def run_experiment(model_name,experiment_config):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data

    transformations = basic_transforms(augmentation_type=None, model_type=model_name)

    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    #!!! only for testing
    train_dataset = get_subset(train_dataset, fraction=0.01)
    val_dataset = get_subset(val_dataset, fraction=0.01)
    test_dataset = get_subset(test_dataset, fraction=0.01)
    
    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=None,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    best_val_acc = 0
    vals = []
    for epoch in range(5):

        train(model, train_loader, optimizer, criterion, device)
        val_acc = validate(model, val_loader, device)

        vals.append(val_acc)
    best_val_acc = max(vals)
    mean_acc = mean(vals)
    std_acc = stdev(vals)
    test_acc = validate(model, test_loader, device)

    return best_val_acc, mean_acc, std_acc, test_acc

In [ ]:
results = []

for i, config in enumerate(experiment_grid):


    print(config)

    model = "ResNet"

    best_val_score, mean_acc, std_acc, test_score = run_experiment(model,config)
    results.append({
        "model": model,
        "config": config,
        "val_acc": best_val_score,
        "test_acc": test_score,
        "mean_acc": mean_acc,
        "std_acc": std_acc
    })

### Tutaj wybieramy najlepsze konfiguracje (jedna na kazdy model) i zaczynamy testowanie augmentacji

## Augmentation techniques


In [33]:
# przykladowo
best_conf_resnet = {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}

In [ ]:
# standard operations
flip_transforms = basic_transforms(augmentation_type="flip")
shift_transforms = basic_transforms(augmentation_type="shift")
rotation_transforms = basic_transforms(augmentation_type="rotation")
# basic_augmentations = {
#     "flip": flip_transforms,
#     "shift": shift_transforms, 
#     "rotation": rotation_transforms}
basic_augmentations = ["flip", "shift", "rotation"]
# more advanced data augmentation
# cutmix 

In [35]:
def run_experiment_augmentation(model_name,experiment_config,augmentation=None, cutmix_collate_fn=None):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data
    transformations = basic_transforms(augmentation_type=augmentation, model_type=model_name)


    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)

    #!!! only for testing
    train_dataset = get_subset(train_dataset, fraction=0.01)
    val_dataset = get_subset(val_dataset, fraction=0.01)
    test_dataset = get_subset(test_dataset, fraction=0.01)
    

    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=cutmix_collate_fn,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    best_val_acc = 0
    vals = []
    for epoch in range(2):

        train(model, train_loader, optimizer, criterion, device)
        val_acc = validate(model, val_loader, device)

        vals.append(val_acc)
    best_val_acc = max(vals)
    mean_acc = mean(vals)
    std_acc = stdev(vals)
    test_acc = validate(model, test_loader, device)

    return best_val_acc, mean_acc, std_acc, test_acc

In [38]:
# augmentation experiments

results = []

for augm in basic_augmentations:

    model = "ResNet"

    best_val_score, mean_acc, std_acc, test_score = run_experiment_augmentation(model,best_conf_resnet, augm, cutmix_collate_fn=None)
    results.append({
        "model": model,
        "augmentation": augm,
        "val_acc": best_val_score,
        "test_acc": test_score,
        "mean_acc": mean_acc,
        "std_acc": std_acc
    })

best_val_score, mean_acc, std_acc, test_score = run_experiment_augmentation(model,best_conf_resnet, None, cutmix_collate_fn=cutmix_collate_fn)
results.append({
        "model": model,
        "augmentation": "cutmix",
        "val_acc": best_val_score,
        "test_acc": test_score,
        "mean_acc": mean_acc,
        "std_acc": std_acc
    })

100%|██████████| 113/113 [00:17<00:00,  6.61it/s]


In [ ]:
# tutaj też sprawdzamy jak się modele zachowywały etc.

In [40]:
from torch.utils.data import Subset
from collections import defaultdict
import random

def few_shot_subset(dataset, samples_per_class=5, seed=42):


    random.seed(seed)

    class_indices = defaultdict(list)

    for idx, label in enumerate(dataset.targets):
        class_indices[label].append(idx)

    selected_indices = []

    for label, indices in class_indices.items():
        if len(indices) < samples_per_class:
            raise ValueError(f"Za mało próbek dla klasy {label}")

        selected = random.sample(indices, samples_per_class)
        selected_indices.extend(selected)

    return Subset(dataset, selected_indices)

## Few-shot learning

In [ ]:
#przyklad jak wybrać podzbior do fewshot learningu, wystarczy na wczytanym datasecie zastosować few_shot_subset

transformations = basic_transforms(augmentation_type=None, model_type='ResNet')

train_dataset = get_train_dataset(transform=transformations)
val_dataset, test_dataset = get_val_train_dataset(model_type='ResNet')

few_shot_train =  few_shot_subset(train_dataset)
few_shot_val =  few_shot_subset(val_dataset)
few_shot_test =  few_shot_subset(test_dataset)